# PyG Benchmark

Run this notebook in the **PyG environment**. It saves results to `comparison_outputs/` so `compare_results.ipynb` can compare them side-by-side with the DGL run.

Stages:
1. Graph construction — node/edge counts per type
2. Forward-pass parity — load DGL weights, compare single-pass scores *(requires `dgl_benchmark.ipynb` to have run first)*
3. Training — 1 pretrain epoch + 5 finetune epochs, save validation metrics
4. Disease-centric evaluation — per-disease AUROC on a sample of test diseases

In [1]:
import sys, os, json, pickle
import torch
import numpy as np

REPO_ROOT   = os.path.abspath(os.path.join(os.getcwd()))
PYG_ROOT    = os.path.join(REPO_ROOT, 'pyg_implementation')
OUT_DIR     = os.path.join(REPO_ROOT, 'comparison_outputs')
os.makedirs(OUT_DIR, exist_ok=True)

sys.path.insert(0, PYG_ROOT)
import txgnn

print('txgnn from:', txgnn.__file__)
print('Output dir:', OUT_DIR)

txgnn from: /home/dsls/Desktop/projects/BardiaSabbagh/txgnn-refactor/pyg_implementation/txgnn/__init__.py
Output dir: /home/dsls/Desktop/projects/BardiaSabbagh/txgnn-refactor/comparison_outputs


In [2]:
# ── Shared config (must match dgl_benchmark.ipynb exactly) ──
DATA_FOLDER   = os.path.join(REPO_ROOT, 'mock')
SPLIT         = 'complex_disease'
SEED          = 42
DEVICE        = 'cuda:0' if torch.cuda.is_available() else 'cpu'
N_HID         = 100
N_INP         = 100
N_OUT         = 100
PROTO         = True
PROTO_NUM     = 5
ATTENTION     = False
SIM_MEASURE   = 'all_nodes_profile'
AGG_MEASURE   = 'rarity'
N_PRETRAIN    = 1
N_FINETUNE    = 5
BATCH_SIZE    = 1024
LR            = 1e-3
SAMPLE_DISEASES = 5

DD_ETYPES = [
    ('drug', 'contraindication', 'disease'),
    ('drug', 'indication', 'disease'),
    ('drug', 'off-label use', 'disease'),
    ('disease', 'rev_contraindication', 'drug'),
    ('disease', 'rev_indication', 'drug'),
    ('disease', 'rev_off-label use', 'drug'),
]

print('Device:', DEVICE)

Device: cuda:0


---
## Stage 1 — Graph construction

In [3]:
torch.manual_seed(SEED)
np.random.seed(SEED)

data = txgnn.TxData(data_folder_path=DATA_FOLDER)
data.prepare_split(split=SPLIT, seed=SEED)
G = data.G

print('Graph loaded')
print('  node_types:', G.node_types)
print('  # edge types:', len(G.edge_types))

Found local copy...
Found local copy...
Found local copy...
Found saved processed KG... Loading...
Splits detected... Loading splits....
Creating PyG graph....
Done!
Graph loaded
  node_types: ['anatomy', 'biological_process', 'cellular_component', 'disease', 'drug', 'effect/phenotype', 'exposure', 'gene/protein', 'molecular_function', 'pathway']
  # edge types: 50


In [4]:
graph_stats = {
    'node_counts': {ntype: G[ntype].num_nodes for ntype in G.node_types},
    'edge_counts': {str(et): G[et].num_edges for et in G.edge_types},
    'edge_indices': {},
}

for et in G.edge_types:
    ei = G[et].edge_index
    src = ei[0].numpy()
    dst = ei[1].numpy()
    idx = np.lexsort((dst, src))
    graph_stats['edge_indices'][str(et)] = {
        'src': src[idx].tolist(),
        'dst': dst[idx].tolist(),
    }

with open(os.path.join(OUT_DIR, 'pyg_graph_stats.json'), 'w') as f:
    json.dump(graph_stats, f)

print('Stage 1 saved: pyg_graph_stats.json')
print('  node types:', list(graph_stats['node_counts'].keys()))
print('  edge type count:', len(graph_stats['edge_counts']))

Stage 1 saved: pyg_graph_stats.json
  node types: ['anatomy', 'biological_process', 'cellular_component', 'disease', 'drug', 'effect/phenotype', 'exposure', 'gene/protein', 'molecular_function', 'pathway']
  edge type count: 50


---
## Stage 2 — Forward-pass parity

Load the DGL model weights saved by `dgl_benchmark.ipynb`, copy them into the PyG model (parameter names are identical), run a single forward pass, and save the scores.

**Skip this stage if `dgl_benchmark.ipynb` hasn't been run yet** — move on to Stage 3.

In [5]:
dgl_state_path = os.path.join(OUT_DIR, 'dgl_model_state.pt')
dgl_emb_path   = os.path.join(OUT_DIR, 'dgl_node_emb.pkl')
dgl_available  = os.path.exists(dgl_state_path) and os.path.exists(dgl_emb_path)
print('DGL weights available:', dgl_available)
if not dgl_available:
    print('Skipping forward-pass parity — run dgl_benchmark.ipynb first.')

DGL weights available: True


In [6]:
if dgl_available:
    torch.manual_seed(0)
    np.random.seed(0)

    model_fp = txgnn.TxGNN(data=data, device=DEVICE)
    model_fp.model_initialize(
        n_hid=N_HID, n_inp=N_INP, n_out=N_OUT,
        proto=PROTO, proto_num=PROTO_NUM,
        attention=ATTENTION,
        sim_measure=SIM_MEASURE,
        agg_measure=AGG_MEASURE,
    )

    # Load DGL weights into PyG model
    dgl_state = torch.load(dgl_state_path, map_location='cpu')
    pyg_state = model_fp.model.state_dict()

    common   = set(dgl_state.keys()) & set(pyg_state.keys())
    only_dgl = set(dgl_state.keys()) - set(pyg_state.keys())
    only_pyg = set(pyg_state.keys()) - set(dgl_state.keys())

    print(f'Common params: {len(common)}')
    if only_dgl: print(f'Only in DGL:   {sorted(only_dgl)}')
    if only_pyg: print(f'Only in PyG:   {sorted(only_pyg)}')

    shape_errors = []
    new_state = dict(pyg_state)
    with torch.no_grad():
        for name in common:
            if dgl_state[name].shape == pyg_state[name].shape:
                new_state[name] = dgl_state[name].clone()
            else:
                shape_errors.append((name, dgl_state[name].shape, pyg_state[name].shape))

    if shape_errors:
        print('Shape mismatches (skipped):')
        for n, sd, sp in shape_errors:
            print(f'  {n}: DGL={sd}  PyG={sp}')
    else:
        print('All weights copied successfully')

    model_fp.model.load_state_dict(new_state, strict=False)

    # Sync node embeddings
    with open(dgl_emb_path, 'rb') as f:
        dgl_emb = pickle.load(f)
    with torch.no_grad():
        for ntype in G.node_types:
            if ntype in dgl_emb:
                data.G[ntype].inp = dgl_emb[ntype].clone()
                model_fp.G[ntype].inp = dgl_emb[ntype].clone()
    print('Node embeddings synced from DGL')

Common params: 202
All weights copied successfully
Node embeddings synced from DGL


In [14]:
if dgl_available:
    from txgnn.utils import Full_Graph_NegSampler

    torch.manual_seed(1)
    G_dev = data.G.to(DEVICE)
    neg_sampler = Full_Graph_NegSampler(G_dev, 1, 'fix_dst', DEVICE)
    neg_G = neg_sampler(G_dev)

    model_fp.model.eval()
    with torch.no_grad():
        scores_pos, scores_neg, pos_out, neg_out = model_fp.model(
            G_dev, neg_G, pretrain_mode=False, mode='test'
        )

    forward_scores = {
        str(et): scores_pos[et].detach().cpu().tolist()
        for et in DD_ETYPES
        if et in scores_pos
    }

    with open(os.path.join(OUT_DIR, 'pyg_forward_scores.json'), 'w') as f:
        json.dump(forward_scores, f)

    print('Stage 2 saved: pyg_forward_scores.json')
    for et, scores in forward_scores.items():
        print(f'  {et}: {len(scores)} edges, mean score={float(np.mean(scores)):.4f}')

Stage 2 saved: pyg_forward_scores.json
  ('drug', 'contraindication', 'disease'): 26640 edges, mean score=0.0859
  ('drug', 'indication', 'disease'): 7740 edges, mean score=-0.3551
  ('drug', 'off-label use', 'disease'): 2150 edges, mean score=0.2496
  ('disease', 'rev_contraindication', 'drug'): 26640 edges, mean score=0.3045
  ('disease', 'rev_indication', 'drug'): 7740 edges, mean score=-0.1118
  ('disease', 'rev_off-label use', 'drug'): 2150 edges, mean score=0.4228


---
## Stage 3 — Training metrics

In [8]:
torch.manual_seed(0)
np.random.seed(0)

model = txgnn.TxGNN(data=data, device=DEVICE)
model.model_initialize(
    n_hid=N_HID, n_inp=N_INP, n_out=N_OUT,
    proto=PROTO, proto_num=PROTO_NUM,
    attention=ATTENTION,
    sim_measure=SIM_MEASURE,
    agg_measure=AGG_MEASURE,
)
print('Model initialized, params:', sum(p.numel() for p in model.model.parameters()))

Model initialized, params: 1015000


In [9]:
torch.manual_seed(0)
model.pretrain(
    n_epoch=N_PRETRAIN,
    learning_rate=LR,
    batch_size=BATCH_SIZE,
    train_print_per_n=9999,
)
print('Pretrain done')

Creating minibatch pretraining dataloader...
Start pre-training with #param: 1015000
Epoch: 0 Step: 0 LR: 0.00100 Loss 0.6939, Pretrain Micro AUROC 0.4826 Pretrain Micro AUPRC 0.4933 Pretrain Macro AUROC 0.4926 Pretrain Macro AUPRC 0.6109
Pretrain done


In [10]:
torch.manual_seed(0)
model.finetune(
    n_epoch=N_FINETUNE,
    learning_rate=LR,
    train_print_per_n=9999,
    valid_per_n=N_FINETUNE,
)
print('Finetune done')

Epoch: 0 LR: 0.00100 Loss 0.7474, Train Micro AUROC 0.4899 Train Micro AUPRC 0.5029 Train Macro AUROC 0.5220 Train Macro AUPRC 0.5237
----- AUROC Performance in Each Relation -----
('drug', 'contraindication', 'disease'): 0.4582468349543237
('drug', 'indication', 'disease'): 0.5162577953381541
('drug', 'off-label use', 'disease'): 0.5356646836127636
('disease', 'rev_contraindication', 'drug'): 0.5044263583685036
('disease', 'rev_indication', 'drug'): 0.5091265298559782
('disease', 'rev_off-label use', 'drug'): 0.608292806922661
----- AUPRC Performance in Each Relation -----
('drug', 'contraindication', 'disease'): 0.47542611764558695
('drug', 'indication', 'disease'): 0.5163147837347064
('drug', 'off-label use', 'disease'): 0.5405075003579334
('disease', 'rev_contraindication', 'drug'): 0.49820905309025726
('disease', 'rev_indication', 'drug'): 0.5396148291019025
('disease', 'rev_off-label use', 'drug'): 0.5721992727173161
----------------------------------------------
Validation.....


In [11]:
from txgnn.utils import evaluate_fb

(auroc_rel, auprc_rel, micro_auroc, micro_auprc, macro_auroc, macro_auprc), loss = \
    evaluate_fb(model.best_model, model.g_valid_pos, model.g_valid_neg,
                data.G.to(DEVICE), DD_ETYPES, DEVICE)

metrics = {
    'macro_auroc': macro_auroc,
    'macro_auprc': macro_auprc,
    'micro_auroc': micro_auroc,
    'micro_auprc': micro_auprc,
    'loss': loss,
    'auroc_per_etype': {str(k): v for k, v in auroc_rel.items()},
    'auprc_per_etype': {str(k): v for k, v in auprc_rel.items()},
}

with open(os.path.join(OUT_DIR, 'pyg_metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)

print('Stage 3 saved: pyg_metrics.json')
print(f'  Macro AUROC={macro_auroc:.4f}  Macro AUPRC={macro_auprc:.4f}  Loss={loss:.4f}')

Stage 3 saved: pyg_metrics.json
  Macro AUROC=0.5130  Macro AUPRC=0.5181  Loss=0.7125


---
## Stage 4 — Disease-centric evaluation

In [13]:
evaluator = txgnn.TxEval(model=model)
disease_ids = evaluator.retrieve_disease_idxs_test_set('indication')[:SAMPLE_DISEASES]
print('Evaluating diseases:', disease_ids.tolist())

result = evaluator.eval_disease_centric(
    disease_idxs=disease_ids.tolist(),
    relation='indication',
    return_raw=True,
    show_plot=False,
    verbose=False,
    simulate_random=False,
)

disease_auroc = {str(k): float(v) for k, v in result['result']['AUROC'].items()}

with open(os.path.join(OUT_DIR, 'pyg_disease_auroc.json'), 'w') as f:
    json.dump({'disease_ids': disease_ids.tolist(), 'auroc': disease_auroc}, f, indent=2)

print('Stage 4 saved: pyg_disease_auroc.json')
for did, auc in disease_auroc.items():
    print(f'  disease {did}: AUROC={auc:.4f}')

Evaluating diseases: [7701.0, 12661.0, 11195.0, 16569.0, 6343.0]


  0%|          | 0/5 [00:00<?, ?it/s]

Stage 4 saved: pyg_disease_auroc.json
  disease 20751_5469_15914_21272: AUROC=0.5942
  disease 5027.0: AUROC=0.6821
  disease 3996.0: AUROC=0.4176
  disease 9417_18555: AUROC=0.4361
  disease 19032.0: AUROC=0.0483


---
All outputs saved to `comparison_outputs/`. Open `compare_results.ipynb` to see the side-by-side comparison.